1. Environment Setup

This notebook prepares the environment for our Mini-BART research project.

In [ ]:
# %pip install -q "numpy==1.26.4" "transformers<5" datasets evaluate rouge_score sentencepiece accelerate

In [ ]:
# %pip install -q "scipy==1.15.3"

In [ ]:
# %pip install -q "numpy==2.2.6" "transformers==4.57.6" "datasets==3.6.0" evaluate rouge_score sentencepiece accelerate tensorboard pandas matplotlib pyyaml

2. Imports

Import the libraries required for this project.

In [ ]:
import numpy as np
import scipy
import torch
import transformers

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

3. GPU Verification

Verify that PyTorch can access our Colab GPU.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import torch
import numpy as np
import scipy
import transformers
import datasets
import evaluate
import huggingface_hub
import fsspec

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nNumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)
print("\nAll imports successful!")

4. Configuration

Define project-wide settings.

## 4. Loading the Pretrained BART Teacher

We use facebook/bart-large-cnn as our teacher model.

It is already pretrained and fine-tuned for CNN/DailyMail summarization.

Our objective is to use this teacher to guide the training of a smaller BART student through knowledge distillation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

teacher = teacher.to("cuda")
teacher.eval()

print("Teacher loaded successfully!")
print("Model:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in teacher.parameters()))

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create a permanent storage location
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
os.makedirs(TEACHER_DIR, exist_ok=True)

# Save the model and tokenizer
teacher.save_pretrained(TEACHER_DIR)
tokenizer.save_pretrained(TEACHER_DIR)

print("Teacher model and tokenizer saved successfully!")
print("Location:", TEACHER_DIR)


In [ ]:
print(os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_path = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

test_tokenizer = AutoTokenizer.from_pretrained(teacher_path)
test_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_path)

print("Reload successful!")
print("Parameters:", sum(p.numel() for p in test_model.parameters()))

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

print("Teacher directory exists:", os.path.exists(TEACHER_DIR))
print("Files:", os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    TEACHER_DIR
).to("cuda")

teacher.eval()

print("Teacher loaded successfully!")
print("Parameters:", sum(p.numel() for p in teacher.parameters()))
print("Device:", next(teacher.parameters()).device)

In [ ]:
article = """
The Indian Space Research Organisation (ISRO) successfully launched
its latest Earth observation satellite aboard a Polar Satellite Launch
Vehicle (PSLV) from the Satish Dhawan Space Centre in Sriharikota.
The satellite will provide high-resolution images to support
agriculture, disaster management, urban planning, and environmental
monitoring. Scientists stated that the mission demonstrates India's
continued progress in developing reliable space technology.
"""

inputs = tokenizer(
    article,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to("cuda")

with torch.no_grad():
    summary_ids = teacher.generate(
        **inputs,
        max_length=100,
        num_beams=4,
        forced_bos_token_id=0
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Teacher summary:\n", summary)

In [ ]:

from datasets import load_dataset

# Load 5 examples from the test split
dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)


In [ ]:
import datasets
import huggingface_hub

print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

In [ ]:
sample = dataset[0]

print("ARTICLE:\n")
print(sample["article"][:1500])

print("\nREFERENCE SUMMARY:\n")
print(sample["highlights"])

In [ ]:

import torch
teacher.eval()
teacher_results = []
for i, example in enumerate(dataset):

    article = example["article"]
    reference = example["highlights"]

    # Tokenize the article
    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    # Generate summary
    with torch.no_grad():
        summary_ids = teacher.generate(
            **inputs,
            max_length=128,
            min_length=30,
            num_beams=4,
            forced_bos_token_id=0
        )

    # Convert generated tokens into text
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    teacher_results.append({
        "id": example["id"],
        "article": article,
        "reference_summary": reference,
        "teacher_summary": generated_summary
    })

    print(f"Completed article {i + 1}/5")

print("\nAll teacher summaries generated!")


In [ ]:
for i, result in enumerate(teacher_results):

    print(f"\n{'='*80}")
    print(f"ARTICLE {i+1}")
    print(f"{'='*80}")

    print("\nREFERENCE SUMMARY:\n")
    print(result["reference_summary"])

    print("\nTEACHER SUMMARY:\n")
    print(result["teacher_summary"])

In [ ]:
import json
import os

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/mini_bart/"
    "teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_baseline_5_samples.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        teacher_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved to:", output_path)

In [ ]:
for i, result in enumerate(teacher_results):
    print(f"\nArticle {i+1}")
    print("ID:", result["id"])
    print("Article length:", len(result["article"]))
    print("Reference:", result["reference_summary"][:150])
    print("Teacher:", result["teacher_summary"])

In [ ]:
print("Total results:", len(teacher_results))

print("\nArticle 5")
print("ID:", teacher_results[4]["id"])
print("Article length:", len(teacher_results[4]["article"]))

print("\nReference summary:")
print(teacher_results[4]["reference_summary"])

print("\nTeacher summary:")
print(teacher_results[4]["teacher_summary"])

In [ ]:
%pip install -q evaluate rouge_score

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
print("ROUGE metric loaded successfully!")

In [ ]:
predictions = [
    result["teacher_summary"]
    for result in teacher_results
]

references = [
    result["reference_summary"]
    for result in teacher_results
]

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("Teacher ROUGE Scores:\n")

for metric, score in rouge_scores.items():
    print(f"{metric}: {score:.4f}")

MEASURING EXTRACTIVENESS (E0)

Unigram overlap: copied individual words

Bigram overlap: copied consecutive two-word phrases

4-gram overlap: copied consecutive four-word phrases

copy rate= summary n-grams found in source/total summary n grams

Novel rate = 1-copy rate

In [ ]:
import re
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())
def get_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]
def source_overlap(source, summary, n):
    source_tokens = tokenize(source)
    summary_tokens = tokenize(summary)

    source_ngrams = set(get_ngrams(source_tokens, n))
    summary_ngrams = get_ngrams(summary_tokens, n)

    if not summary_ngrams:
        return 0.0

    copied = sum(
        ngram in source_ngrams
        for ngram in summary_ngrams
    )

    return copied / len(summary_ngrams)

In [ ]:
copy_rates = {1: [], 2: [], 4: []}
for result in teacher_results:

    # Reconstruct the exact truncated article input
    encoded = tokenizer(
        result["article"],
        max_length=1024,
        truncation=True
    )

    source_text = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    summary = result["teacher_summary"]

    for n in [1, 2, 4]:
        rate = source_overlap(source_text, summary, n)
        copy_rates[n].append(rate)


print("Teacher Source Overlap:\n")

for n in [1, 2, 4]:
    avg_copy = sum(copy_rates[n]) / len(copy_rates[n])
    avg_novel = 1 - avg_copy

    print(f"{n}-gram Copy Rate:  {avg_copy:.4f}")
    print(f"{n}-gram Novel Rate: {avg_novel:.4f}\n")

SAVING BASELINE RESULT

will compare it with E1 AND E2 LATER

SAVING ROUGE METRICS and EXTRACTIVENESS RECORD IN GOOGLE DRIVE

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),
    "rouge": rouge_scores,
    "source_overlap": {
        f"{n}-gram": {
            "copy_rate": sum(copy_rates[n]) / len(copy_rates[n]),
            "novel_rate": 1 - sum(copy_rates[n]) / len(copy_rates[n])
        }
        for n in [1, 2, 4]
    }
}

RESULTS_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher_baseline"
os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(RESULTS_DIR, "teacher_metrics.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher baseline metrics saved!")
print(output_path)

In [ ]:
import os

print("Current working directory:", os.getcwd())

print("\nChecking project directory:")

for path in [
    "projects/01-mini-bart/src/metrics.py",
    "/content/Research/projects/01-mini-bart/src/metrics.py"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])

In [ ]:
import os
research_path = "/content/drive/MyDrive/Research"

print("Research folder exists:", os.path.exists(research_path))

if os.path.exists(research_path):
    print("\nContents:")
    print(os.listdir(research_path))

VS Code (Windows)
Source of truth, Write and edit metrics.py, notebooks and other project files.

Google Drive
Make the repository files accessible to Colab.

Colab T4
Import and execute the Python modules remotely.

cloning REPO IN DRIVE

In [ ]:
#cloning repo in drive
%cd /content/drive/MyDrive

!git clone https://github.com/HeavenlyDemon27/Research.git

verifying the cloning

In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/Research"

print("Repository exists:", os.path.exists(PROJECT_ROOT))
print("Project files:", os.listdir(PROJECT_ROOT))

Sync the latest code to the remote runtime

This updates the Drive copy with the metrics.py I just pushed.

In [ ]:
%cd /content/drive/MyDrive/Research
!git pull --ff-only origin main

Import our functions

We're importing our own reusable functions rather than keeping all evaluation logic inside the noteboo

In [ ]:
import sys
SRC_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from metrics import compute_rouge, compute_source_overlap

print("Metrics module imported successfully!")

Testing module against our teacher results

In [ ]:
predictions = [r["teacher_summary"] for r in teacher_results]
references = [r["reference_summary"] for r in teacher_results]
sources = []
for r in teacher_results:
    encoded = tokenizer(
        r["article"],
        max_length=1024,
        truncation=True
    )

    source = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    sources.append(source)

# Calculate metrics using our reusable module
rouge_scores = compute_rouge(predictions, references)

overlap_scores = compute_source_overlap(
    sources,
    predictions
)

print("ROUGE:", rouge_scores)
print("\nSource overlap:", overlap_scores)

ABOVE there, What are we checking?

We're checking whether the functions in metrics.py reproduce the evaluation results we previously calculated directly in the notebook.

Expected ROUGE results should be approximately:

Metric	Previous result

ROUGE-1	0.4674

ROUGE-2	0.2629

ROUGE-L	0.3522

ROUGE-Lsum	0.4082

Small differences may arise from implementation or rounding, but the results should match if the same inputs and settings are used

FINISH E0- SAVING FINAL TEACHERS METRICS

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),

    "rouge": {
        key: float(value)
        for key, value in rouge_scores.items()
    },

    "source_overlap": overlap_scores
}

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/"
    "mini_bart/teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_metrics.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher metrics saved!")
print(output_path)

VERIFY

In [ ]:
#verifying
print(os.listdir(RESULTS_DIR))

E0 EVALUATION COMPLETED. YAAAAYYYYY🥳🥳

E1 starting here

making sure of libraries once again

In [ ]:
!pip install -q datasets transformers accelerate

import our functions

uploading my local data.py  to colab

verifying GPU again

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected.")

verifying libraries

In [ ]:
%pip install evaluate

In [ ]:

import torch
import transformers
import datasets
import accelerate
import evaluate

print("Environment verification")
print("-" * 40)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Evaluate:", evaluate.__version__)

# Actual GPU computation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(2048, 2048, device=device)
y = torch.randn(2048, 2048, device=device)
z = x @ y

torch.cuda.synchronize()

print("\nGPU:", torch.cuda.get_device_name(0))
print("Test tensor device:", z.device)
print("Test computation successful:", torch.isfinite(z).all().item())
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GB"
)


connecting drive(got confused and messed up)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Mini-BART"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "logs"), exist_ok=True)

print("Project storage ready:", PROJECT_DIR)
print("Checkpoints:", os.path.join(PROJECT_DIR, "checkpoints"))
print("Logs:", os.path.join(PROJECT_DIR, "logs"))


checking location of project in drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart"

print("Project exists:", os.path.exists(PROJECT_DIR))
print("\nProject contents:")

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

    if level >= 2:
        dirs[:] = []
        continue

    for file in files:
        print("  " * (level + 1) + file)

checking if train.py and model.py are there in drive if not then its not syncrhonised with local and github


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart"

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

print("\n--- train.py ---")
with open("src/train.py", "r", encoding="utf-8") as f:
    print(f.read())

print("\n--- model.py ---")
with open("src/model.py", "r", encoding="utf-8") as f:
    print(f.read())

synchronizing Drive

In [ ]:
%cd /content/drive/MyDrive/Research/projects/01-mini-bart

!git pull origin main

the above failed. so nvm it

In [ ]:
import shutil

source = "projects/01-mini-bart/src/data.py"
backup = "projects/01-mini-bart/src/data_drive_backup.py"

shutil.copy2(source, backup)

print("Backup created:", backup)

In [ ]:
%cd /content/drive/MyDrive/Research

!git status
!git stash push -m "Preserve Drive data.py before GitHub sync" -- projects/01-mini-bart/src/data.py

In [ ]:
!git pull --ff-only origin main

In [ ]:
!git status

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Research/projects/01-mini-bart")

for filename in ["src/model.py", "src/train.py", "src/data.py"]:
    path = project / filename
    print(f"\n{filename}")
    print("Exists:", path.exists())
    print("Size:", path.stat().st_size if path.exists() else "Missing")
    print(path.read_text()[:200] if path.exists() else "")

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Research/projects/01-mini-bart"
)

sys.path.insert(0, str(PROJECT_DIR))

from src.model import (
    build_student_model,
    initialize_student_from_teacher,
)

from src.train import (
    create_optimizer_and_scheduler,
    train_one_epoch,
)

print("All imports successful!")

print("\nModel functions:")
print(build_student_model)
print(initialize_student_from_teacher)

print("\nTraining functions:")
print(create_optimizer_and_scheduler)
print(train_one_epoch)

inspecting synchronized env

In [ ]:
import os
import subprocess

RESEARCH_DIR = "/content/drive/MyDrive/Research"
PROJECT_DIR = os.path.join(RESEARCH_DIR, "projects/01-mini-bart")

os.chdir(RESEARCH_DIR)

print("Project directory:", PROJECT_DIR)
print("\nLatest Git commit:")
subprocess.run(["git", "log", "-1", "--oneline"])

print("\nGit status:")
subprocess.run(["git", "status", "--short"])

print("\nRequired files:")
files = [
    "src/data.py",
    "src/model.py",
    "src/train.py",
    "src/distillation.py",
    "src/evaluate.py",
    "src/metrics.py",
    "notebooks/00_setup.ipynb",
]

for file in files:
    path = os.path.join(PROJECT_DIR, file)
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {file}")

print("\nTraining functions:")
with open(os.path.join(PROJECT_DIR, "src/train.py")) as f:
    for line in f:
        if line.startswith("def ") or line.startswith("class "):
            print(line.strip())

print("\nData functions:")
with open(os.path.join(PROJECT_DIR, "src/data.py")) as f:
    for line in f:
        if line.startswith("def ") or line.startswith("class "):
            print(line.strip())

inspecting source code 1)train.py

In [ ]:
from pathlib import Path

train_file = Path(PROJECT_DIR) / "src/train.py"

print("=" * 70)
print("CURRENT src/train.py")
print("=" * 70)

print(train_file.read_text())

inspecting data.py

In [ ]:
data_file = Path(PROJECT_DIR) / "src/data.py"

print("=" * 70)
print("CURRENT src/data.py")
print("=" * 70)

print(data_file.read_text())

DOWN IS HOW YOU PULL FROM GITHUB TO GOOGLE DRIVE!!

In [ ]:
import os
import subprocess

RESEARCH_DIR = "/content/drive/MyDrive/Research"

os.chdir(RESEARCH_DIR)

# Check repository status first
subprocess.run(["git", "status"], check=True)

# Pull latest committed code
result = subprocess.run(
    ["git", "pull", "origin", "main"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Git pull failed. Resolve this before continuing.")

In [ ]:
from pathlib import Path

train_path = Path(
    "/content/drive/MyDrive/Research/projects/01-mini-bart/src/train.py"
)

print("Exists:", train_path.exists())
print("Size:", train_path.stat().st_size, "bytes")

print("\nFunctions:")
for line in train_path.read_text().splitlines():
    if line.startswith("def "):
        print(line)

In [ ]:
import shutil

drive_path = "/content/drive/MyDrive"

total, used, free = shutil.disk_usage(drive_path)

print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used: {used / 1024**3:.2f} GB")
print(f"Free: {free / 1024**3:.2f} GB")

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

RESEARCH_DIR = Path("/content/drive/MyDrive/Research")
PROJECT_DIR = RESEARCH_DIR / "projects/01-mini-bart"
SRC_DIR = PROJECT_DIR / "src"

assert (RESEARCH_DIR / ".git").exists(), (
    f"Git repository not found at {RESEARCH_DIR}. "
    "Do not run the stale Drive copy; clone the repository first."
)

# Never overwrite modified tracked files. Untracked backups are left untouched.
tracked_changes = subprocess.run(
    ["git", "diff", "--quiet"],
    cwd=RESEARCH_DIR,
).returncode != 0
staged_changes = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=RESEARCH_DIR,
).returncode != 0

if tracked_changes or staged_changes:
    status = subprocess.run(
        ["git", "status", "--short"],
        cwd=RESEARCH_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout
    raise RuntimeError(
        "Drive repository has modified/staged tracked files. "
        "Nothing was overwritten.\\n"
        + status
        + "\\nReview or commit/stash these changes, then rerun this cell."
    )

untracked = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=RESEARCH_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if untracked:
    print("Leaving untracked files untouched:")
    print(untracked)

branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=RESEARCH_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if branch != "main":
    raise RuntimeError(
        f"Expected branch 'main', found {branch!r}. "
        "Switch to main deliberately before syncing."
    )

subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    cwd=RESEARCH_DIR,
    check=True,
)

assert (SRC_DIR / "train.py").exists()
assert (SRC_DIR / "data.py").exists()
assert (SRC_DIR / "model.py").exists()

# Reload modules from the freshly synchronized Drive source.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import src.data
import src.model
import src.train

importlib.reload(src.data)
importlib.reload(src.model)
importlib.reload(src.train)

from src.data import (
    load_cnn_dailymail,
    prepare_dataset,
    create_dataloaders,
    StatefulRandomSampler,
)
from src.model import (
    build_student_model,
    initialize_student_from_teacher,
)
from src.train import train_model

print("Repository synchronized:", RESEARCH_DIR)
print("Branch:", branch)
print("train.py:", (SRC_DIR / "train.py").stat().st_size, "bytes")
print("data.py:", (SRC_DIR / "data.py").stat().st_size, "bytes")
print("model.py:", (SRC_DIR / "model.py").stat().st_size, "bytes")
print("Imported modules:", src.data.__file__, src.model.__file__, src.train.__file__)


checking files, storage etc


In [ ]:
from pathlib import Path
import torch
import os

# 1. Check project files
SRC_DIR = PROJECT_DIR / "src"

required_files = [
    "data.py",
    "model.py",
    "train.py",
    "metrics.py",
    "evaluate.py",
]

print("PROJECT FILES")
for filename in required_files:
    path = SRC_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

# 2. Check likely teacher checkpoint locations
teacher_candidates = [
    Path("/content/drive/MyDrive/research_artifacts/mini_bart/teacher"),
    Path("/content/drive/MyDrive/Research/artifacts/teacher"),
    Path("/content/drive/MyDrive/Research/teacher"),
]

print("\nTEACHER CHECKPOINT")
for path in teacher_candidates:
    print(f"\n{path}")
    if path.exists():
        print("FOUND")
        print("Contents:", [p.name for p in path.iterdir()][:15])
    else:
        print("Not found")

# 3. GPU status
print("\nGPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# 4. Drive free space
stat = os.statvfs("/content/drive/MyDrive")
free_gb = stat.f_bavail * stat.f_frsize / (1024**3)
print("\nApprox. Drive free space:", round(free_gb, 2), "GB")

In [ ]:
import gc
import torch
from transformers import BartForConditionalGeneration, AutoTokenizer

from src.model import (
    build_student_model,
    initialize_student_from_teacher,
)

# Paths
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
device = torch.device("cuda")

# 1. Load the pretrained teacher on CPU
print("Loading teacher...")
teacher = BartForConditionalGeneration.from_pretrained(TEACHER_DIR)
teacher.eval()

# 2. Build the 6-encoder / 6-decoder student
print("Building Mini-BART...")
student = build_student_model(teacher)

# 3. Transfer selected teacher weights into the student
print("Transferring teacher weights...")
initialize_student_from_teacher(teacher, student)

# 4. Load tokenizer from the same local checkpoint
tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

# 5. Release teacher from memory — E1 does not need it during training
del teacher
gc.collect()

# 6. Move student to the T4
student = student.to(device)
student.train()

# 7. Report architecture and parameter counts
student_params = sum(p.numel() for p in student.parameters())

print("\n========== MINI-BART STATUS ==========")
print("Device:", next(student.parameters()).device)
print("Encoder layers:", student.config.encoder_layers)
print("Decoder layers:", student.config.decoder_layers)
print(f"Student parameters: {student_params:,}")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("Tokenizer vocabulary:", len(tokenizer))
print("======================================")

loading dataset

In [ ]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import load_cnn_dailymail

raw_dataset = load_cnn_dailymail()

print("\nDataset loaded successfully!")
for split, data in raw_dataset.items():
    print(f"{split}: {len(data):,} examples")

taking some examples


In [ ]:
import os
import torch
from src.data import (
    load_cnn_dailymail,
    prepare_dataset,
    create_dataloaders,
)

# Keep temporary dataset cache on Colab storage, not Google Drive
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"

print("Loading CNN/DailyMail v3.0.0...")
raw_dataset = load_cnn_dailymail()

print("\nOriginal dataset sizes:")
for split, data in raw_dataset.items():
    print(f"{split}: {len(data):,} examples")

print("\nPreparing benchmark subset...")
tokenized_dataset = prepare_dataset(
    raw_dataset,
    tokenizer,
    max_train_samples=128,
    max_val_samples=32,
    max_test_samples=0,
)

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=2,
    num_workers=0,
)

print("\n========== DATASET BENCHMARK ==========")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Test examples:", len(tokenized_dataset["test"]))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

print("=======================================")

This cell performs forward propagation, cross-entropy loss calculation, backward propagation, gradient clipping, and one optimizer update.used FP16 mixed precision, matching our planned training configuration

In [ ]:
import time
import torch
from torch.optim import AdamW

# Prepare model
student.train()
student.zero_grad(set_to_none=True)

# One batch
batch = next(iter(train_loader))
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

# Optimizer for benchmark only
optimizer = AdamW(
    student.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler("cuda")

# Reset GPU memory statistics
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Benchmark
torch.cuda.synchronize()
start_time = time.time()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    outputs = student(**batch)
    loss = outputs.loss

scaler.scale(loss).backward()

scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(
    student.parameters(),
    max_norm=1.0,
)

scaler.step(optimizer)
scaler.update()

torch.cuda.synchronize()
elapsed = time.time() - start_time

# Results
peak_memory = torch.cuda.max_memory_allocated() / 1024**3
reserved_memory = torch.cuda.max_memory_reserved() / 1024**3

print("\n========== GPU TRAINING BENCHMARK ==========")
print(f"Loss: {loss.item():.4f}")
print(f"Step time: {elapsed:.2f} seconds")
print(f"Peak allocated GPU memory: {peak_memory:.2f} GB")
print(f"Peak reserved GPU memory: {reserved_memory:.2f} GB")
print(f"Remaining GPU memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")
print("Optimizer update: completed")
print("============================================")

In [ ]:
import time
import gc
import torch
from torch.optim import AdamW
from src.data import create_dataloaders

for batch_size in [4, 8]:

    print(f"\nTesting batch size: {batch_size}")

    gc.collect()
    torch.cuda.empty_cache()
    student.zero_grad(set_to_none=True)

    try:
        train_loader_test, _, _ = create_dataloaders(
            tokenized_dataset,
            tokenizer,
            batch_size=batch_size,
            num_workers=0,
        )

        batch = next(iter(train_loader_test))
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer_test = AdamW(
            student.parameters(),
            lr=5e-5,
            weight_decay=0.01,
        )

        scaler_test = torch.amp.GradScaler("cuda")

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start = time.time()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = student(**batch)
            loss = outputs.loss

        scaler_test.scale(loss).backward()
        scaler_test.unscale_(optimizer_test)

        torch.nn.utils.clip_grad_norm_(
            student.parameters(), 1.0
        )

        scaler_test.step(optimizer_test)
        scaler_test.update()

        torch.cuda.synchronize()
        elapsed = time.time() - start

        print(f"Loss: {loss.item():.4f}")
        print(f"Step time: {elapsed:.2f} seconds")
        print(
            f"Peak GPU memory: "
            f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
        )
        print(f"Time per example: {elapsed / batch_size:.3f} seconds")

        del optimizer_test, scaler_test, batch
        del train_loader_test

    except torch.cuda.OutOfMemoryError:
        print("OUT OF MEMORY — this batch size is not viable.")
        student.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()

print("\nBatch-size benchmark complete.")

In [ ]:
import importlib
import src.train

# Reload the latest version from Google Drive
importlib.reload(src.train)

print("Available functions:")
for name in [
    "create_optimizer_and_scheduler",
    "train_one_epoch",
    "validate_one_epoch",
    "save_checkpoint",
    "train_model",
]:
    print(f"{name}: {hasattr(src.train, name)}")

In [ ]:
import shutil
import inspect
from pathlib import Path

from src.train import save_checkpoint

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return "Not found"

    total = sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    )
    return f"{total / 1024**3:.2f} GB"

print("========== STORAGE ==========")

print("Drive:")
!df -h /content/drive

print("\nTeacher checkpoint size:")
print(folder_size_gb(
    "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
))

print("\nProject checkpoint folder:")
print(folder_size_gb(PROJECT_DIR / "checkpoints"))

print("\n========== CHECKPOINT FUNCTION ==========")
print(inspect.getsource(save_checkpoint))

In [ ]:
import inspect
from pathlib import Path
from src.train import train_model, create_optimizer_and_scheduler

print("========= TRAINING FUNCTION =========")
print(inspect.signature(train_model))
print(inspect.getsource(train_model))

print("\n========= DATALOADERS =========")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n========= BATCH CONFIGURATION =========")
batch = next(iter(train_loader))
for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

print("\n========= MODEL =========")
print("Device:", next(student.parameters()).device)
print("Parameters:", sum(p.numel() for p in student.parameters()))

print("\n========= CHECKPOINT DIRECTORY =========")
OUTPUT_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/student_baseline"
print("Output:", OUTPUT_DIR)
print("Exists:", Path(OUTPUT_DIR).exists())

In [ ]:
import inspect
from src.data import prepare_dataset, create_dataloaders

print("========= RAW DATASET =========")
for split in raw_dataset:
    print(split, len(raw_dataset[split]))

print("\n========= PREPROCESSING SIGNATURE =========")
print(inspect.signature(prepare_dataset))

print("\n========= DATALOADER SIGNATURE =========")
print(inspect.signature(create_dataloaders))

print("\n========= PREPROCESSING IMPLEMENTATION =========")
print(inspect.getsource(prepare_dataset))

In [ ]:
import importlib
import inspect
from pathlib import Path
import src.train

# Refresh the module to ensure we're using the latest source
importlib.reload(src.train)

save_checkpoint = src.train.save_checkpoint

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return "Not found"

    total = sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    )
    return f"{total / 1024**3:.2f} GB"

print("========== STORAGE ==========")

print("Drive:")
!df -h /content/drive

print("\nColab local storage:")
!df -h /content

print("\nTeacher checkpoint size:")
print(folder_size_gb(
    "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
))

print("\nProject checkpoint folder:")
print(folder_size_gb(Path(PROJECT_DIR) / "checkpoints"))

print("\nProject artifacts folder:")
print(folder_size_gb(Path(PROJECT_DIR) / "artifacts"))

print("\n========== CHECKPOINT FUNCTION ==========")
print(inspect.getsource(save_checkpoint))

In [ ]:
import inspect
from src.train import train_model

print(inspect.getsource(train_model))

# **FULL DATASET PREPROCESSING**

In [ ]:
import os
import gc
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
from src.data import create_dataloaders

# Reuse the tokenized data saved to Drive yesterday. No dataset download or tokenization.
TOKENIZED_DIR = Path("/content/drive/MyDrive/MiniBART_Research/E1/tokenized_cnn_dm_v1")
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

assert TOKENIZED_DIR.exists(), f"Saved tokenized dataset missing: {TOKENIZED_DIR}"

print("Loading tokenized CNN/DailyMail from Drive...")
tokenized_dataset = load_from_disk(str(TOKENIZED_DIR))

# The test split was intentionally not tokenized. Create an empty split with matching features.
assert "train" in tokenized_dataset and "validation" in tokenized_dataset
tokenized_dataset["test"] = tokenized_dataset["validation"].select([])

EXPECTED_TRAIN = 287_113
EXPECTED_VALIDATION = 13_368
assert len(tokenized_dataset["train"]) == EXPECTED_TRAIN
assert len(tokenized_dataset["validation"]) == EXPECTED_VALIDATION
assert len(tokenized_dataset["test"]) == 0

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

print("Recreating fresh DataLoaders...")
train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=8,
    num_workers=0,
)

# Confirm that no sample has been consumed by inspection or benchmarking.
assert train_loader.sampler.position == 0
assert train_loader.sampler.permutation is None
assert len(train_loader) == 35_890
assert len(val_loader) == 1_671
assert len(test_loader) == 0

# Inspect a batch WITHOUT iterating the stateful training sampler.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)
sample_batch = data_collator([tokenized_dataset["train"][i] for i in range(8)])

print("\\n========== DATASET READY (LOADED FROM DRIVE) ==========")
print(f"Train examples: {len(train_loader.dataset):,}")
print(f"Validation examples: {len(val_loader.dataset):,}")
print(f"Test examples: {len(test_loader.dataset):,}")
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
for key, value in sample_batch.items():
    print(f"{key}: {tuple(value.shape)}")
print("Sampler position:", train_loader.sampler.position)
print("Sampler permutation initialized:", train_loader.sampler.permutation is not None)
print("No training batch consumed during inspection.")
del sample_batch
gc.collect()


diagnostic cell below

In [ ]:
import torch

print("========== PRE-TRAINING CHECK ==========")

print("\nGPU:")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Allocated memory:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\nDataset objects:")
for name, obj in list(globals().items()):
    if any(term in name.lower() for term in
           ["dataset", "tokenized", "train_loader", "val_loader", "test_loader"]):
        try:
            print(f"{name}: {type(obj).__name__}, length={len(obj)}")
        except Exception:
            print(f"{name}: {type(obj).__name__}")

print("\nDataLoader sizes:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

print("\nGPU memory available:",
      round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

In [ ]:
print("========== TOKENIZED DATASET INSPECTION ==========")

for split, data in tokenized_dataset.items():
    print(f"\nSplit: {split}")
    print("Type:", type(data).__name__)

    try:
        print("Examples:", len(data))
    except Exception as e:
        print("Length unavailable:", e)

    if hasattr(data, "column_names"):
        print("Columns:", data.column_names)
    elif isinstance(data, dict):
        print("Keys:", list(data.keys()))

print("\nDataLoader batch counts:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

**SAVING training and validation splits**

In [ ]:

import os
from google.colab import drive
from datasets import DatasetDict, load_from_disk

# 1. Ensure Google Drive is mounted
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# 2. Define a dedicated E1 storage location
SAVE_DIR = (
    "/content/drive/MyDrive/MiniBART_Research/"
    "E1/tokenized_cnn_dm_v1"
)

# Never overwrite an existing saved dataset
if os.path.exists(SAVE_DIR):
    raise FileExistsError(
        f"Destination already exists: {SAVE_DIR}\n"
        "Choose a new versioned folder before saving."
    )

# 3. Build a DatasetDict from the existing in-memory datasets
datasets_to_save = DatasetDict({
    "train": tokenized_dataset["train"],
    "validation": tokenized_dataset["validation"],
})

# 4. Save to Google Drive
os.makedirs(os.path.dirname(SAVE_DIR), exist_ok=True)

print("Saving tokenized datasets to Google Drive...")
datasets_to_save.save_to_disk(SAVE_DIR)

print("\nSave completed.")
print("Location:", SAVE_DIR)

# 5. Reload from Drive to verify persistence
print("\nVerifying saved datasets...")
reloaded_dataset = load_from_disk(SAVE_DIR)

for split in ["train", "validation"]:
    original_count = len(datasets_to_save[split])
    reloaded_count = len(reloaded_dataset[split])

    print(
        f"{split}: original={original_count:,}, "
        f"reloaded={reloaded_count:,}"
    )

    assert original_count == reloaded_count

# 6. Verify sample data
assert reloaded_dataset["train"][0]["input_ids"] == \
       tokenized_dataset["train"][0]["input_ids"]

print("\nSUCCESS: Dataset saved and verified in Google Drive.")


# **Reinitialize Mini-BART and test a full-length batch**

In [ ]:

import gc
import time
import torch
from torch.optim import AdamW
from transformers import BartForConditionalGeneration

# Release the benchmark student
del student
gc.collect()
torch.cuda.empty_cache()

# Reinitialize from the original teacher
print("Loading teacher...")
teacher = BartForConditionalGeneration.from_pretrained(TEACHER_DIR)
teacher.eval()

print("Building and initializing Mini-BART...")
student = build_student_model(teacher)
initialize_student_from_teacher(teacher, student)

del teacher
gc.collect()

student = student.to(device)
student.train()

print("Mini-BART reinitialized successfully.")

# Test one actual full-length batch
batch = next(iter(train_loader))
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

optimizer_test = AdamW(
    student.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)

scaler_test = torch.amp.GradScaler("cuda")

student.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.time()

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = student(**batch)
    loss = outputs.loss

scaler_test.scale(loss).backward()
scaler_test.unscale_(optimizer_test)

torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)

scaler_test.step(optimizer_test)
scaler_test.update()

torch.cuda.synchronize()
elapsed = time.time() - start

print("\n========== FULL-LENGTH BATCH TEST ==========")
print(f"Input shape: {tuple(batch['input_ids'].shape)}")
print(f"Label shape: {tuple(batch['labels'].shape)}")
print(f"Loss: {loss.item():.4f}")
print(f"Step time: {elapsed:.2f} seconds")
print(
    f"Peak GPU memory: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

# Clean up temporary test optimizer and gradients
del optimizer_test, scaler_test, outputs, batch
student.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

print("\nFull-length batch test completed.")


# **Launch E1 training**

In [ ]:
import gc
import os
from pathlib import Path
import torch
from transformers import BartForConditionalGeneration

# Re-import from the synchronized source, not an earlier notebook binding.
import importlib
import src.data
import src.model
import src.train
importlib.reload(src.data)
importlib.reload(src.model)
importlib.reload(src.train)

from src.model import build_student_model, initialize_student_from_teacher
from src.train import train_model

# Reconstruct loaders immediately before training to reset the stateful sampler.
# Earlier benchmark/inspection cells may have iterated over the old loader.
from src.data import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

assert train_loader.sampler.position == 0
assert train_loader.sampler.permutation is None
assert len(train_loader) == 35_890
assert len(val_loader) == 1_671
assert len(test_loader) == 0

# Hard checks: this must be the full E1 training/validation dataset,
# not the 128-example benchmark subset from the earlier notebook cells.
EXPECTED_TRAIN = 287_113
EXPECTED_VALIDATION = 13_368
BATCH_SIZE = 8
NUM_EPOCHS_E1 = 2

assert len(train_loader.dataset) == EXPECTED_TRAIN, (
    f"Expected full train split ({EXPECTED_TRAIN:,}), "
    f"found {len(train_loader.dataset):,}. Rerun full preprocessing cell 110."
)
assert len(val_loader.dataset) == EXPECTED_VALIDATION, (
    f"Expected full validation split ({EXPECTED_VALIDATION:,}), "
    f"found {len(val_loader.dataset):,}. Rerun full preprocessing cell 110."
)
assert train_loader.batch_size == BATCH_SIZE, (
    f"Expected batch size {BATCH_SIZE}, found {train_loader.batch_size}."
)
assert train_loader.num_workers == 0, "Exact sampler resume requires num_workers=0."
assert hasattr(train_loader.sampler, "state_dict"), (
    "Training loader does not have the stateful sampler."
)
assert torch.cuda.is_available(), "CUDA GPU is not available."
device = torch.device("cuda")

TEACHER_DIR = Path("/content/drive/MyDrive/research_artifacts/mini_bart/teacher")
OUTPUT_DIR = Path("/content/drive/MyDrive/research_artifacts/mini_bart/E1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LATEST_CHECKPOINT = OUTPUT_DIR / "latest_checkpoint.pt"
BEST_CHECKPOINT = OUTPUT_DIR / "best_checkpoint.pt"

print("Existing E1 checkpoint files:", [
    p.name for p in OUTPUT_DIR.glob("*.pt")
])

# If only a best checkpoint exists, do not silently start over and risk
# overwriting a prior experiment's best model.
if BEST_CHECKPOINT.exists() and not LATEST_CHECKPOINT.exists():
    raise RuntimeError(
        "best_checkpoint.pt exists but latest_checkpoint.pt is missing. "
        "Inspect the E1 directory before deciding whether to resume or restart."
    )

# Resume only from a structurally valid, compatible latest checkpoint.
RESUME_FROM = None
if LATEST_CHECKPOINT.exists():
    checkpoint = torch.load(
        LATEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "epoch", "global_step", "model_state_dict",
        "optimizer_state_dict", "scheduler_state_dict",
        "scaler_state_dict", "sampler_state_dict", "rng_state", "config",
    }
    missing = required_keys.difference(checkpoint)
    if missing:
        raise RuntimeError(
            f"Latest checkpoint is incomplete (missing {sorted(missing)}). "
            "It has been preserved; do not overwrite it."
        )

    config = checkpoint["config"]
    if (
        int(checkpoint["epoch"]) > 0
        and "history" not in checkpoint
    ):
        raise RuntimeError(
            "Checkpoint has completed epochs but does not contain their "
            "loss history. It is preserved; refusing to produce an incomplete "
            "final training record by resuming it."
        )
    if config.get("objective") != "supervised_cross_entropy_only":
        raise RuntimeError("Checkpoint objective is not E1 supervised CE.")
    if config.get("num_epochs") != NUM_EPOCHS_E1:
        raise RuntimeError(
            f"Checkpoint was created for {config.get('num_epochs')} epochs, "
            f"but this run is configured for {NUM_EPOCHS_E1}. "
            "Do not resume with a different scheduler horizon."
        )

    sampler_state = checkpoint["sampler_state_dict"]
    sampler_position = int(sampler_state.get("position", 0))
    saved_permutation = sampler_state.get("permutation")
    if (
        saved_permutation is not None
        and len(saved_permutation) != len(train_loader.dataset)
    ):
        raise RuntimeError(
            "Checkpoint sampler permutation belongs to a different "
            "training-set size. It is preserved; refusing an unsafe resume."
        )
    if sampler_position < 0 or (
        saved_permutation is not None
        and sampler_position > len(saved_permutation)
    ):
        raise RuntimeError("Checkpoint sampler position is invalid.")

    is_mid_epoch = (
        sampler_state.get("permutation") is not None
        and sampler_position > 0
    )

    # Older mid-epoch checkpoints lack the accumulated loss counters.
    # They can restore optimizer progress, but cannot report a true full-epoch
    # training loss, so refuse automatic resume rather than record a false metric.
    if is_mid_epoch and not {
        "epoch_loss_sum", "epoch_examples"
    }.issubset(checkpoint):
        raise RuntimeError(
            "Found a legacy mid-epoch checkpoint without full-epoch loss "
            "accumulators. It is preserved, but cannot produce a valid "
            "full-epoch training-loss record. Do not resume it automatically."
        )

    RESUME_FROM = str(LATEST_CHECKPOINT)
    print(
        "Compatible checkpoint found:",
        f"epoch={checkpoint['epoch'] + 1}",
        f"global_step={checkpoint['global_step']}",
        f"sampler_position={sampler_position}",
    )
    del checkpoint
else:
    print("No latest E1 checkpoint found. Starting a fresh E1 run.")

print("\n========== E1 CONFIGURATION ==========")
print(f"Training examples: {len(train_loader.dataset):,}")
print(f"Validation examples: {len(val_loader.dataset):,}")
print(f"Train batches per epoch: {len(train_loader):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS_E1}")
print("Learning rate: 5e-5")
print("Objective: supervised cross-entropy only")
print("Checkpoint directory:", OUTPUT_DIR)
print("Resume:", RESUME_FROM or "Fresh initialization")
print("GPU:", torch.cuda.get_device_name(0))

# Always construct the same teacher-derived architecture.
# If RESUME_FROM exists, train_model immediately restores the checkpoint's
# model/optimizer/scheduler/scaler/sampler/RNG states over this initialization.
gc.collect()
torch.cuda.empty_cache()

print("\nLoading teacher checkpoint...")
teacher = BartForConditionalGeneration.from_pretrained(str(TEACHER_DIR))
teacher.eval()

print("Building and initializing Mini-BART...")
student = build_student_model(teacher)
initialize_student_from_teacher(teacher, student)

del teacher
gc.collect()
torch.cuda.empty_cache()

student = student.to(device)
student.train()

print("Student ready:", sum(p.numel() for p in student.parameters()), "parameters")

history = train_model(
    model=student,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    output_dir=str(OUTPUT_DIR),
    num_epochs=NUM_EPOCHS_E1,
    resume_from=RESUME_FROM,
)

print("\n========== E1 TRAINING CALL FINISHED ==========")
print(history)


# **DIAGNOSTIC**

In [ ]:
import torch
from pathlib import Path
import os

print("========== GPU STATUS ==========")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

print("\n========== ENVIRONMENT ==========")
print("Working directory:", os.getcwd())

print("\n========== GOOGLE DRIVE ==========")
drive = Path("/content/drive/MyDrive")
print("Drive mounted:", drive.exists())

if drive.exists():
    for folder in ["MiniBART_Research", "research_artifacts/mini_bart"]:
        path = drive / folder
        print(f"\n{path}:")
        if path.exists():
            for item in path.iterdir():
                print(" ", item.name)
        else:
            print(" Not found")

mount G-drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

project and saved checkpoints


In [ ]:
import os

print("CONTENT DIRECTORY:")
print(os.listdir("/content"))

print("\nGOOGLE DRIVE:")
print(os.listdir("/content/drive/MyDrive"))

print("\nMINIBART CHECKPOINTS:")
base = "/content/drive/MyDrive/MiniBART_Research"

if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        for f in files:
            if f.endswith((".pt", ".pth")):
                print(os.path.join(root, f))
else:
    print("MiniBART_Research folder not found")

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

folders = [
    drive_root / "Research",
    drive_root / "Mini-BART",
    drive_root / "research_artifacts",
    drive_root / "MiniBART_Research",
]

print("========== PROJECT FILES ==========")

for folder in folders:
    print(f"\n{folder}")

    if not folder.exists():
        print("  Not found")
        continue

    for name in ["train.py", "train_kd.py", "data.py", "model.py"]:
        matches = list(folder.rglob(name))
        for path in matches[:5]:
            print(" ", path)

print("\n========== CHECKPOINTS ==========")

for folder in folders:
    if not folder.exists():
        continue

    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in (".pt", ".pth"):
            print(f"{path} | {path.stat().st_size / 1024**3:.2f} GB")

print("\n========== SEARCH COMPLETE ==========")

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Research/projects/01-mini-bart")

print("========== PROJECT STRUCTURE ==========")
for folder in ["src", "configs", "experiments", "tests"]:
    p = PROJECT / folder
    print(f"\n{folder}/")
    if p.exists():
        for item in sorted(p.iterdir()):
            print(" ", item.name, "/" if item.is_dir() else "")
    else:
        print("  Not found")

print("\n========== E1 TRAINER: src/train.py ==========")
train_file = PROJECT / "src" / "train.py"
print(train_file.read_text())

print("\n========== PROJECT CONFIG FILES ==========")
for p in PROJECT.rglob("*.json"):
    if ".cache" not in str(p):
        print(f"\n--- {p.relative_to(PROJECT)} ---")
        print(p.read_text()[:12000])

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Research/projects/01-mini-bart")

print("========== TRAINING ENTRY POINTS ==========")

for folder in [PROJECT, PROJECT / "src"]:
    for file in folder.glob("*.py"):
        content = file.read_text(encoding="utf-8")
        if "train_model(" in content or "build_student_model(" in content:
            print(f"\n--- {file.relative_to(PROJECT)} ---")
            lines = content.splitlines()

            for i, line in enumerate(lines):
                if "train_model(" in line or "build_student_model(" in line:
                    start = max(0, i - 15)
                    end = min(len(lines), i + 35)
                    print("\n".join(
                        f"{j+1}: {lines[j]}"
                        for j in range(start, end)
                    ))

print("\n========== DATA LOADER CONFIGURATION ==========")

data_file = PROJECT / "src" / "data.py"
lines = data_file.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "def create_dataloaders" in line or "class StatefulRandomSampler" in line:
        start = i
        end = min(len(lines), i + 100)
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(start, end)
        ))

>   **KAGGLE NOTEBOOK VERIFICATION/AUDIT STARTS HERE** 

In [ ]:
import sys
import os
import torch

print("=" * 60)
print("SYSTEM")
print("=" * 60)
print("Python:", sys.version)
print("Working directory:", os.getcwd())

print("\n" + "=" * 60)
print("PYTORCH & GPU")
print("=" * 60)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")
        print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: Running without CUDA GPU")

print("\n" + "=" * 60)
print("DEPENDENCIES")
print("=" * 60)

from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch", "transformers", "datasets", "accelerate",
    "evaluate", "numpy", "pyarrow", "tokenizers",
    "rouge_score", "tensorboard", "tqdm"
]

for package in packages:
    try:
        print(f"{package:15} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:15} NOT INSTALLED")

In [ ]:
%pip install evaluate==0.4.6

In [ ]:
import torch

assert torch.cuda.is_available()

for i in range(torch.cuda.device_count()):
    device = torch.device(f"cuda:{i}")
    x = torch.randn(1000, 1000, device=device)
    y = x @ x

    torch.cuda.synchronize(i)

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Computation successful: {y.shape}")
    print(
        "Allocated memory:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 3),
        "GB"
    )

> **setting up working directories on KAGGLE**

In [ ]:
import os

BASE_DIR = "/kaggle/working/research"

directories = [
    f"{BASE_DIR}/projects/01-mini-bart/data",
    f"{BASE_DIR}/projects/01-mini-bart/experiments/E0",
    f"{BASE_DIR}/projects/01-mini-bart/experiments/E1/checkpoints",
    f"{BASE_DIR}/projects/01-mini-bart/experiments/E2/checkpoints",
    f"{BASE_DIR}/projects/01-mini-bart/experiments/E3/checkpoints",
    f"{BASE_DIR}/cache",
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("Workspace created.")

for directory in directories:
    print(directory)

CONFIGURE HUGGING FACE CACHE

In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/research/cache/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/research/cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/research/cache/transformers"

os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)

print("Hugging Face cache configured.")

GETTING OUR PROJECT CODE IN KAGGLE

In [ ]:
%cd /kaggle/working

!git clone https://github.com/HeavenlyDemon27/Research.git Research

**INSPECTING**

In [ ]:
!ls -la /kaggle/working/Research
!find /kaggle/working/Research -maxdepth 3 -type f | head -60

#  TEST DATASET ACCESS

In [ ]:
from datasets import load_dataset

DATASET_ID = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"

dataset_stream = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    streaming=True
)

samples = list(dataset_stream.take(5))

print("Number of samples:", len(samples))
print("\nAvailable fields:", samples[0].keys())

print("\nSample article:")
print(samples[0]["article"][:500])

print("\nSample summary:")
print(samples[0]["highlights"])

In [ ]:
REPO_DIR = "/kaggle/working/Research"

PROJECT_DIR = f"{REPO_DIR}/projects/01-mini-bart"

print("Repository exists:", os.path.exists(REPO_DIR))
print("Mini-BART exists:", os.path.exists(PROJECT_DIR))

!ls -la /kaggle/working/Research/projects/01-mini-bart

> **LIST SOURCE FILE**

In [ ]:
!find /kaggle/working/Research/projects/01-mini-bart/src -maxdepth 3 -type f

> **LIST CONFIGURATION**

In [ ]:
!find /kaggle/working/Research/projects/01-mini-bart/configs -maxdepth 3 -type f

> **LIST EXPERIMENT FILES**

In [ ]:
!find /kaggle/working/Research/projects/01-mini-bart/experiments -maxdepth 4 -type f

In [ ]:
%cd /kaggle/working/Research/projects/01-mini-bart #baseline configuration

!cat configs/baseline.yaml

In [ ]:
!sed -n '1,280p' src/train.py #training script

In [ ]:
!sed -n '1,280p' src/data.py #data pipeline

In [ ]:
!sed -n '1,300p' src/utils.py #checkpoints and utility functions

In [ ]:
!cat experiments/experiment_registry.csv #experiment registry

> **FIXING WORKING DIRECTORY**

In [ ]:
import os

PROJECT_DIR = "/kaggle/working/Research/projects/01-mini-bart"

os.chdir(PROJECT_DIR)

print("Current directory:", os.getcwd())
print("Project files:", os.listdir("."))

> **ONE FINAL CHECK BEFORE LAUNCHING E1**

In [ ]:
print("========== BASELINE CONFIG ==========")

with open("configs/baseline.yaml") as f:
    print(f.read())

print("\n========== E1 TRAINING SCRIPT ==========")

with open("src/train.py") as f:
    print(f.read())

# LAUNCHING E1 TRAINING I GUESS???

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/Research/projects/01-mini-bart")

print("========== PROJECT NOTEBOOKS ==========")
for p in (root / "notebooks").glob("*"):
    print(p.name)

print("\n========== TRAINING ENTRY-POINT SEARCH ==========")

terms = [
    "train_model(",
    "DataLoader(",
    "BartForConditionalGeneration",
    "student_encoder_layers",
    "load_dataset(",
]

for p in root.rglob("*"):
    if not p.is_file() or p.suffix not in {".py", ".ipynb"}:
        continue
    if ".ipynb_checkpoints" in str(p) or ".git" in str(p):
        continue

    try:
        lines = p.read_text(errors="ignore").splitlines()
    except Exception:
        continue

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, 1)
        if any(term in line for term in terms)
    ]

    if matches:
        print(f"\n--- {p.relative_to(root)} ---")
        for i, line in matches:
            print(f"{i}: {line[:200]}")

In [ ]:
from pathlib import Path

print("========== KAGGLE INPUTS ==========")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\n========== WORKING DIRECTORY ==========")
for p in Path("/kaggle/working").iterdir():
    print(p)

print("\n========== TEACHER CHECKPOINT ==========")
teacher = Path("/kaggle/working/Research/projects/01-mini-bart")
print("Project exists:", teacher.exists())

print("\n========== TOKENIZED DATA ==========")
for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for p in base.rglob("tokenized_cnn_dm_v1"):
        print("Found:", p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/Research/projects/01-mini-bart")

print("=" * 70)
print("BASELINE CONFIG")
print("=" * 70)

config_path = root / "configs" / "baseline.yaml"

if config_path.exists():
    print(config_path.read_text())
else:
    print("baseline.yaml NOT FOUND")

print("\n" + "=" * 70)
print("E1 TRAINING FUNCTION")
print("=" * 70)

train_path = root / "src" / "train.py"

lines = train_path.read_text().splitlines()

terms = [
    "def train_model",
    "NUM_EPOCHS",
    "num_epochs",
    "gradient_accumulation",
    "accumulation_steps",
    "optimizer.step",
    "scheduler.step",
    "save_checkpoint",
    "validate",
]

for i, line in enumerate(lines):
    if any(term in line for term in terms):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")

print("\n" + "=" * 70)
print("E1 RESUME TEST STATUS")
print("=" * 70)

test_path = root / "tests" / "test_interrupted_vs_uninterrupted_e1.py"
print("Test exists:", test_path.exists())

if test_path.exists():
    print(test_path.read_text())

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/Research/projects/01-mini-bart")

def show_section(path, start, end, title):
    lines = path.read_text().splitlines()

    print(f"\n{'=' * 70}")
    print(title)
    print(f"{'=' * 70}")

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


show_section(
    root / "src/train.py",
    1, 175,
    "TRAINING CONFIGURATION AND TRAINING LOOP"
)

show_section(
    root / "src/train.py",
    360, 510,
    "COMPLETE train_model FUNCTION"
)

show_section(
    root / "src/data.py",
    280, 360,
    "DATALOADER CONSTRUCTION"
)

print("\n" + "=" * 70)
print("CONFIGURATION LOADING REFERENCES")
print("=" * 70)

for path in root.rglob("*.py"):
    if ".git" in str(path):
        continue

    lines = path.read_text(errors="ignore").splitlines()

    for i, line in enumerate(lines, 1):
        if any(term in line.lower() for term in [
            "baseline.yaml",
            "safe_load",
            "yaml.load",
            "gradient_accumulation_steps",
        ]):
            print(f"{path.relative_to(root)}:{i}: {line.strip()}")

In [7]:
%cd /kaggle/working/Research
!git pull --ff-only origin main

/kaggle/working/Research
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 876 bytes | 876.00 KiB/s, done.
From https://github.com/HeavenlyDemon27/Research
 * branch            main       -> FETCH_HEAD
   5867cf1..c3d6869  main       -> origin/main
Updating 5867cf1..c3d6869
Fast-forward
 .../tests/test_interrupted_vs_uninterrupted_e1.py   | 21 +++++++++++++++++----
 1 file changed, 17 insertions(+), 4 deletions(-)


In [8]:
%cd /kaggle/working/Research/projects/01-mini-bart

/kaggle/working/Research/projects/01-mini-bart


In [5]:
!pip install -q pytest

In [9]:
!python -m pytest -q tests/test_interrupted_vs_uninterrupted_e1.py

.                                                                        [100%]
=============================== warnings summary ===============================
tests/test_interrupted_vs_uninterrupted_e1.py::test_interrupted_resume_matches_uninterrupted_training
  /kaggle/working/Research/projects/01-mini-bart/src/train.py:248: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

tests/test_interrupted_vs_uninterrupted_e1.py::test_interrupted_resume_matches_uninterrupted_training
  /kaggle/working/Research/projects/01-mini-bart/src/train.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
    with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16):

tests/test_interrupted_vs_uninterrupted_e1.py::test_interrupted_resume_matches_uninterrupted_training
  /kaggle/working/Re

# E1 TRAINING TEST, DATALOADERS, TOKENIZATION 

SYNC REPO

In [19]:
%cd /kaggle/working/Research

!git pull --ff-only origin main

%cd /kaggle/working/Research/projects/01-mini-bart

!python -m pytest -q tests/test_interrupted_vs_uninterrupted_e1.py

/kaggle/working/Research
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 3.07 KiB | 785.00 KiB/s, done.
From https://github.com/HeavenlyDemon27/Research
 * branch            main       -> FETCH_HEAD
   c3d6869..8d76e85  main       -> origin/main
Updating c3d6869..8d76e85
Fast-forward
 projects/01-mini-bart/src/data.py | 274 +++++++++++++++-----------------------
 1 file changed, 109 insertions(+), 165 deletions(-)
/kaggle/working/Research/projects/01-mini-bart
.                                                                        [100%]
=============================== warnings summary ===============================
tests/test_interrupted_vs_uninterrupted_e1.py::test_interrupted_resume_matches_uninterrupted_training
  /kaggle/working/Research/projects/01-mini-bart/src/train.py:248: FutureWarning: 

verify env and gpu

In [11]:
import torch
import transformers
import datasets
import sys
import os

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU unavailable"

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
CUDA: True
GPU: Tesla T4
GPU memory: 14.56 GB


RELOAD THE LATEST MODULE

In [12]:
import importlib
import src.data
import src.model
import src.train

importlib.reload(src.data)
importlib.reload(src.model)
importlib.reload(src.train)

from src.data import create_dataloaders
from src.model import prepare_e1_student
from src.train import train_model

print("Latest project modules imported.")

Latest project modules imported.


Load and verify the full tokenized dataset

In [20]:
from pathlib import Path
from datasets import DatasetDict, load_from_disk

from src.data import (
    load_cnn_dailymail,
    load_bart_tokenizer,
    prepare_dataset,
    create_dataloaders,
)

DATA_DIR = Path(
    "/kaggle/working/mini_bart_tokenized_512_128_v1"
)

if DATA_DIR.exists():
    raise FileExistsError(
        f"{DATA_DIR} already exists. Inspect it before overwriting."
    )

print("1. Loading CNN/DailyMail 3.0.0...")
raw_dataset = load_cnn_dailymail()

assert len(raw_dataset["train"]) == 287_113
assert len(raw_dataset["validation"]) == 13_368

print("2. Loading BART tokenizer...")
tokenizer = load_bart_tokenizer()

print("3. Tokenizing train and validation...")
tokenized = prepare_dataset(
    raw_dataset,
    tokenizer,
)

assert len(tokenized["train"]) == 287_113
assert len(tokenized["validation"]) == 13_368
assert len(tokenized["test"]) == 0

print("4. Saving reusable tokenized dataset...")
saved_dataset = DatasetDict({
    "train": tokenized["train"],
    "validation": tokenized["validation"],
})

saved_dataset.save_to_disk(str(DATA_DIR))

print("5. Reloading and verifying...")
tokenized_dataset = load_from_disk(str(DATA_DIR))

assert len(tokenized_dataset["train"]) == 287_113
assert len(tokenized_dataset["validation"]) == 13_368

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=1,
    num_workers=0,
)

assert len(train_loader) == 287_113
assert len(val_loader) == 13_368
assert len(test_loader) == 0

print("\nDATA PIPELINE VERIFIED")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))
print("Saved at:", DATA_DIR)

1. Loading CNN/DailyMail 3.0.0...


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

... (15 outputs left out)

Tokenizing train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Tokenizing test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

AssertionError: 

In [22]:
%cd /kaggle/working/Research
!git pull --ff-only origin main

%cd /kaggle/working/Research/projects/01-mini-bart

import importlib
import src.data as data

importlib.reload(data)

train_loader, val_loader, test_loader = data.create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=1,
    num_workers=0,
)

assert len(train_loader) == 287_113
assert len(val_loader) == 13_368
assert len(test_loader) == 0

print("DATA PIPELINE VERIFIED")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

/kaggle/working/Research
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 659 bytes | 659.00 KiB/s, done.
From https://github.com/HeavenlyDemon27/Research
 * branch            main       -> FETCH_HEAD
   8d76e85..eca2fb2  main       -> origin/main
Updating 8d76e85..eca2fb2
Fast-forward
 projects/01-mini-bart/src/data.py | 7 ++++---
 1 file changed, 4 insertions(+), 3 deletions(-)
/kaggle/working/Research/projects/01-mini-bart
DATA PIPELINE VERIFIED
Train: 287113
Validation: 13368
Test: 0


# SAVING DATASETS CHECKPOINTS

In [24]:
import shutil

shutil.make_archive(
    "/kaggle/working/mini_bart_tokenized_512_128_v1",
    "zip",
    "/kaggle/working/mini_bart_tokenized_512_128_v1"
)

print("Dataset archive created.")

Dataset archive created.


In [34]:
#DOWNLOADING THE DATASET
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

print("Current directory:", os.getcwd())
print("ZIP exists:", os.path.exists("mini_bart_tokenized_512_128_v1.zip"))

display(FileLink("mini_bart_tokenized_512_128_v1.zip"))

Current directory: /kaggle/working
ZIP exists: True


/kaggle/working/mini_bart_tokenized_512_128_v1.zip

In [30]:
!ls -lh /kaggle/working/mini_bart_tokenized_512_128_v1.zip
!ls -lh /kaggle/working/mini_bart_E1_backup.zip

-rw-r--r-- 1 root root 267M Nov 21 13:59 /kaggle/working/mini_bart_tokenized_512_128_v1.zip
ls: cannot access '/kaggle/working/mini_bart_E1_backup.zip': No such file or directory


> saving checkpoints
> Your training directory should contain:
mini_bart_E1/
├── latest_checkpoint.pt
├── best_checkpoint.pt
└── training_summary.json

In [29]:
!ls -lh /kaggle/working/mini_bart_E1/

ls: cannot access '/kaggle/working/mini_bart_E1/': No such file or directory


In [32]:
import os

print("Contents of /kaggle/working:")
for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)
    print(item, "<DIR>" if os.path.isdir(path) else f"{os.path.getsize(path) / (1024**2):.2f} MB")

print("\nSearching for E1 checkpoints:")
for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for f in files:
        if f.endswith((".pt", ".pth", ".bin", ".safetensors")):
            print(os.path.join(root, f))

Contents of /kaggle/working:
mini_bart_tokenized_512_128_v1 <DIR>
.virtual_documents <DIR>
Research <DIR>
mini_bart_tokenized_512_128_v1.zip 266.19 MB
research <DIR>
.pytest_cache <DIR>

Searching for E1 checkpoints:


In [36]:
%cd /kaggle/working/Research/projects/01-mini-bart

from pathlib import Path

for filename in [
    "scripts/run_e1_kaggle.py",
    "src/train.py",
]:
    path = Path(filename)
    print(f"\n{'='*20} {filename} {'='*20}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    for number, line in enumerate(
        path.read_text().splitlines(), start=1
    ):
        if any(word in line.lower() for word in [
            "checkpoint", "output_dir", "save_checkpoint",
            "os.makedirs", "mkdir", "torch.save"
        ]):
            print(f"{number}: {line}")

/kaggle/working/Research/projects/01-mini-bart

==================== scripts/run_e1_kaggle.py ====================
31:     CHECKPOINT_FORMAT_VERSION,
59: def validate_existing_checkpoint(path, train_size):
60:     checkpoint = torch.load(
79:     missing = required.difference(checkpoint)
82:             f"Existing latest checkpoint is incomplete: {sorted(missing)}. "
86:     if checkpoint["format_version"] < CHECKPOINT_FORMAT_VERSION:
88:             "Existing checkpoint format is incompatible with the current trainer."
91:     if checkpoint["gradient_accumulation_steps"] != GRADIENT_ACCUMULATION_STEPS:
92:         raise RuntimeError("Checkpoint gradient accumulation does not match E1.")
94:     config = checkpoint["config"]
106:                 f"Checkpoint config mismatch for {key}: "
110:     sampler_state = checkpoint["sampler_state_dict"]
116:             "Checkpoint sampler belongs to a different training-set size."
121:         raise RuntimeError("Checkpoint sampler position is 

# TRAINING

In [23]:
%cd /kaggle/working/Research
!git pull --ff-only origin main

%cd /kaggle/working/Research/projects/01-mini-bart

!python scripts/run_e1_kaggle.py --help

!python -m pytest -q tests/test_interrupted_vs_uninterrupted_e1.py && python scripts/run_e1_kaggle.py \
    --tokenized-dir /kaggle/working/mini_bart_tokenized_512_128_v1 \
    --output-dir /kaggle/working/mini_bart_E1

/kaggle/working/Research
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.68 KiB | 3.68 MiB/s, done.
From https://github.com/HeavenlyDemon27/Research
 * branch            main       -> FETCH_HEAD
   eca2fb2..516d5a6  main       -> origin/main
Updating eca2fb2..516d5a6
Fast-forward
 projects/01-mini-bart/scripts/run_e1_kaggle.py | 299 +++++++++++++++++++++++++
 1 file changed, 299 insertions(+)
 create mode 100644 projects/01-mini-bart/scripts/run_e1_kaggle.py
/kaggle/working/Research/projects/01-mini-bart
Traceback (most recent call last):
  File "/kaggle/working/Research/projects/01-mini-bart/scripts/run_e1_kaggle.py", line 21, in <module>
    from src.data import create_dataloaders
ModuleNotFoundError: No module named 'src'
.                                                                        [100%]
=

In [47]:
%cd /kaggle/working/Research/projects/01-mini-bart

!PYTHONPATH="$PWD" python scripts/run_e1_kaggle.py \
    --tokenized-dir /kaggle/working/mini_bart_tokenized_512_128_v1 \
    --output-dir /kaggle/working/mini_bart_E1

/kaggle/working/Research/projects/01-mini-bart
GPU: Tesla T4
GPU memory: 14.56 GB

Loading saved tokenized train/validation data...
Loading tokenizer...
Creating fresh DataLoaders...
Compatible E1 checkpoint found: epoch=1 optimizer_updates=4500 sampler_position=36032

========== E1 CONFIGURATION ==========
Train examples: 287,113
Validation examples: 13,368
Test examples: 0
Microbatch size: 1
Gradient accumulation: 8
Effective batch size: 8
Epochs: 2
Learning rate: 5e-05
Weight decay: 0.01
Warmup ratio: 0.06
Objective: supervised cross-entropy only
Teacher: facebook/bart-large-cnn
Output: /kaggle/working/mini_bart_E1
Resume: /kaggle/working/mini_bart_E1/latest_checkpoint.pt

Loading teacher on CPU...
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████████████████| 511/511 [00:00<00:00, 1178.99it/s]
Building Mini-BART and transferring teacher layers [0, 2, 4, 6, 8, 10]...
Student parameters: 229,933,056
Starting E1 t

In [49]:
from pathlib import Path
import os
import time

checkpoint_dir = Path("/kaggle/working/mini_bart_E1")

for f in sorted(checkpoint_dir.glob("*.pt")):
    print(f"{f.name} | {f.stat().st_size / (1024**2):.2f} MB")

latest_checkpoint.pt | 2633.15 MB


In [43]:
import time
from pathlib import Path

checkpoint_dir = Path("/kaggle/working/mini_bart_E1")

while True:
    files = list(checkpoint_dir.glob("*.pt"))
    print("\n", time.strftime("%H:%M:%S"))

    for f in files:
        print(f"{f.name}: {f.stat().st_size / (1024**2):.2f} MB")

    time.sleep(60)


 15:18:44
latest_checkpoint.pt: 2633.15 MB

 15:19:44
latest_checkpoint.pt: 2633.15 MB

 15:20:44
latest_checkpoint.pt: 2633.15 MB


KeyboardInterrupt: 

In [44]:
from pathlib import Path

p = Path("/kaggle/working/mini_bart_E1/latest_checkpoint.pt")

print("Exists:", p.exists())
print("Size (GB):", round(p.stat().st_size / (1024**3), 2))

Exists: True
Size (GB): 2.57


In [45]:
import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/working/mini_bart_E1/latest_checkpoint.pt"
)

print("Loading checkpoint metadata...")

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\n========== CHECKPOINT AUDIT ==========")

print("Keys:", list(ckpt.keys()))

for key in [
    "epoch",
    "global_step",
    "best_val_loss",
    "optimizer_updates",
    "microbatch_position",
]:
    if key in ckpt:
        print(f"{key}: {ckpt[key]}")

print("\nCheckpoint successfully loaded.")

Loading checkpoint metadata...

========== CHECKPOINT AUDIT ==========
Keys: ['format_version', 'epoch', 'batch_position', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'best_val_loss', 'epoch_loss_sum', 'epoch_examples', 'history', 'sampler_state_dict', 'gradient_accumulation_steps', 'rng_state', 'config']
epoch: 0
global_step: 4500
best_val_loss: inf

Checkpoint successfully loaded.


In [46]:
print("Epoch:", ckpt["epoch"])
print("Batch position:", ckpt["batch_position"])
print("Global step:", ckpt["global_step"])
print("Gradient accumulation:", ckpt["gradient_accumulation_steps"])

print("\nSampler state:")
for k, v in ckpt["sampler_state_dict"].items():
    if not hasattr(v, "__len__") or isinstance(v, (str, int, float)):
        print(k, ":", v)
    else:
        print(k, ":", type(v).__name__, "length =", len(v))

Epoch: 0
Batch position: 36032
Global step: 4500
Gradient accumulation: 8

Sampler state:
generator_state : Tensor length = 5056
permutation : list length = 287113
position : 36032
epoch : 0


In [50]:
from pathlib import Path
import shutil

src = Path("/kaggle/working/mini_bart_E1")
backup = Path("/kaggle/working/E1_checkpoint_backup")

backup.mkdir(exist_ok=True)

for name in ["latest_checkpoint.pt", "best_checkpoint.pt", "training_summary.json"]:
    file = src / name
    if file.exists():
        shutil.copy2(file, backup / name)
        print(f"Backed up: {name} | {file.stat().st_size / 1e9:.2f} GB")

Backed up: latest_checkpoint.pt | 2.76 GB


In [1]:
!cd /kaggle/working/Research && git status

/bin/bash: line 1: cd: /kaggle/working/Research: No such file or directory


In [52]:
!ls -lh /kaggle/working/mini_bart_E1/

total 2.6G
-rw-r--r-- 1 root root 2.6G Nov 21 18:36 latest_checkpoint.pt


In [55]:
from IPython.display import FileLink, display
import os

os.chdir("/kaggle/working")

display(
    FileLink(
        "mini_bart_E1/latest_checkpoint.pt",
        url_prefix="/files/"
    )
)

/kaggle/working/mini_bart_E1/latest_checkpoint.pt

In [54]:
!ls -lh /kaggle/working/mini_bart_E1/

total 2.6G
-rw-r--r-- 1 root root 2.6G Nov 21 18:36 latest_checkpoint.pt
